# Практика 01. Python, NumPy и pandas: сжатый ликбез

**Версия:** 2026-09-24 (7b54111)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

Вы уже умеете программировать, поэтому здесь нет объяснений, что такое цикл или функция. Задания
сосредоточены на том, что в Python и особенно в NumPy устроено **не так**, как в C++, Java или C#:

- в Python данные копируются редко — переменные ссылаются на объекты;
- в NumPy циклы по элементам — почти всегда ошибка: операции записываются над массивами целиком
  (векторизация), а массивы разной формы автоматически согласуются (broadcasting);
- pandas — это NumPy с подписанными строками и столбцами и с операциями «как в SQL».

Каждое задание — функция, которую нужно дописать, и ячейка с проверками после неё. Если проверки
прошли, ячейка печатает «OK». Там, где сказано «без циклов», проверка следит, чтобы в коде функции
не было `for` и `while`.

Полезные ссылки: [NumPy для начинающих](https://numpy.org/doc/stable/user/absolute_beginners.html),
[10 минут pandas](https://pandas.pydata.org/docs/user_guide/10min.html).

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re
import time

import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


print("Готово")

# Часть 1. Python

## Задание 1.1. Включения списков и срезы

В Python вместо цикла с `append` обычно пишут **включение** (comprehension):
`[f(x) for x in xs if cond(x)]`. Аналогично строятся словари: `{k: v for k, v in pairs}`.
Срезы `xs[start:stop:step]` работают у списков, строк и массивов; отрицательные индексы считаются
с конца: `xs[-1]` — последний элемент, `xs[::-1]` — список в обратном порядке.

Напишите функцию `squares_of_evens(xs)`, которая возвращает **список** квадратов чётных чисел из `xs`
в обратном порядке. Уложитесь в одну строку.

In [ ]:
def squares_of_evens(xs):
    """[1, 2, 3, 4, 6] -> [36, 16, 4]"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
assert squares_of_evens([1, 2, 3, 4, 6]) == [36, 16, 4], "Для [1, 2, 3, 4, 6] ожидается [36, 16, 4]"
assert squares_of_evens([]) == [], "Для пустого списка ожидается пустой список"
assert squares_of_evens([1, 3, 5]) == [], "Если чётных нет, ожидается пустой список"
assert squares_of_evens([-2, 0]) == [0, 4], "Отрицательные и ноль — тоже чётные; порядок — обратный"
assert isinstance(squares_of_evens(range(5)), list), "Функция должна возвращать список и принимать любой итерируемый объект"
print("OK")

## Задание 1.2. Словари и подсчёт

Словарь `dict` — основная структура данных Python. Метод `d.get(key, default)` возвращает значение
по умолчанию, если ключа нет. Для подсчёта есть готовый `collections.Counter`, но здесь напишите
подсчёт сами, со словарём.

Напишите `word_counts(text)`: разбейте строку на слова (`text.lower().split()`) и верните словарь
«слово → сколько раз встретилось».

In [ ]:
def word_counts(text):
    """'a b A' -> {'a': 2, 'b': 1}"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
assert word_counts("a b A") == {"a": 2, "b": 1}, "Регистр не должен учитываться: 'A' и 'a' — одно слово"
assert word_counts("") == {}, "Для пустой строки ожидается пустой словарь"
assert word_counts("мама мыла раму мама") == {"мама": 2, "мыла": 1, "раму": 1}, "Неверный подсчёт для 'мама мыла раму мама'"
print("OK")

## Задание 1.3. Ссылки и изменяемые аргументы

В Python присваивание не копирует объект: после `b = a` обе переменные ссылаются на **один** список,
и `b.append(1)` изменит и `a`. Отсюда классическая ловушка: значение аргумента по умолчанию
вычисляется **один раз** при определении функции. Функция `def f(x, acc=[])` будет накапливать
элементы в одном и том же списке между вызовами.

Напишите `add_item(item, bucket=None)`: если `bucket` не передан, создайте новый пустой список;
добавьте в него `item` и верните его. Переданный список изменять можно — так ведёт себя `append`.

In [ ]:
def add_item(item, bucket=None):
    """add_item(1) -> [1]; add_item(2) -> [2] (а не [1, 2]!)"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
assert add_item(1) == [1], "add_item(1) должен вернуть [1]"
assert add_item(2) == [2], "Второй вызов без bucket вернул не [2]: список по умолчанию общий для всех вызовов"
b = [0]
assert add_item(5, b) is b and b == [0, 5], "Переданный список должен дополняться и возвращаться тот же объект"
print("OK")

## Задание 1.4. zip, sorted и key

`zip(a, b)` идёт по двум последовательностям параллельно, `enumerate(xs)` даёт пары (индекс, элемент),
`sorted(xs, key=..., reverse=True)` сортирует по произвольному ключу.

Напишите `top_k(names, scores, k)`: верните список из `k` имён с наибольшими баллами, от большего
к меньшему. При равных баллах раньше идёт имя, стоящее раньше в исходном списке (`sorted`
устойчив — этим можно воспользоваться).

In [ ]:
def top_k(names, scores, k):
    """top_k(['a', 'b', 'c'], [1, 3, 2], 2) -> ['b', 'c']"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
assert top_k(["a", "b", "c"], [1, 3, 2], 2) == ["b", "c"], "Для баллов [1, 3, 2] два лучших — b и c"
assert top_k(["a", "b", "c"], [5, 5, 1], 2) == ["a", "b"], "При равных баллах порядок исходного списка должен сохраняться"
assert top_k(["a"], [1], 5) == ["a"], "Если k больше числа имён, вернуть все имена"
print("OK")

# Часть 2. NumPy

Массив `np.ndarray` хранит числа одного типа (`dtype`) в непрерывной памяти и имеет форму `shape`.
Арифметика над массивами поэлементная и выполняется в скомпилированном коде — в десятки и сотни раз
быстрее цикла на Python. Посмотрите сами:

In [ ]:
x = rng.normal(size=1_000_000)

start = time.perf_counter()
total = 0.0
for v in x:
    total += v * v
loop_time = time.perf_counter() - start

start = time.perf_counter()
total_np = np.sum(x * x)
numpy_time = time.perf_counter() - start

print(f"цикл: {loop_time:.3f} с, NumPy: {numpy_time:.4f} с, ускорение в {loop_time / numpy_time:.0f} раз")

Основные операции, которые понадобятся дальше:

| Операция | Пример |
|:---|:---|
| создание | `np.array([[1, 2], [3, 4]])`, `np.zeros((n, d))`, `np.arange(5)`, `np.linspace(0, 1, 11)` |
| форма | `X.shape`, `X.reshape(n, -1)`, `X.T`, `x[:, None]` (добавить ось) |
| агрегаты по осям | `X.sum(axis=0)` — по столбцам, `X.mean(axis=1)` — по строкам, `X.argmax(axis=1)` |
| матричное умножение | `A @ B` |
| маски | `X[X > 0]`, `np.where(cond, a, b)` |
| индексация массивом | `X[[0, 2, 5]]`, `X[rows, cols]` |

## Задание 2.1. Агрегаты по осям и broadcasting: стандартизация

**Broadcasting**: если формы массивов различаются, NumPy «растягивает» оси длины 1 (и недостающие оси
слева). Например, `X` формы `(n, d)` минус вектор формы `(d,)` — это вычитание вектора из **каждой
строки**. Правило: формы сравниваются справа налево, размеры должны совпадать или один из них равен 1.

Напишите `standardize(X)`: вычтите из каждого столбца его среднее и поделите на стандартное
отклонение столбца (`X.std(axis=0)`, то есть с делителем $n$). Без циклов. Исходный массив не меняйте.

$$
x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

In [ ]:
def standardize(X):
    """Каждый столбец результата имеет среднее 0 и стандартное отклонение 1."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
X_test = rng.normal(loc=[1, 100, -5], scale=[1, 50, 0.1], size=(200, 3))
X_copy = X_test.copy()
Z = standardize(X_test)
assert Z.shape == X_test.shape, f"Форма результата {Z.shape}, а должна быть {X_test.shape}"
assert np.allclose(Z.mean(axis=0), 0), "Средние столбцов после стандартизации должны быть равны 0"
assert np.allclose(Z.std(axis=0), 1), "Стандартные отклонения столбцов должны быть равны 1 (а не строк!)"
assert np.array_equal(X_test, X_copy), "Функция изменила исходный массив"
assert_no_loops(standardize)
print("OK")

## Задание 2.2. Маски и копии

Сравнение массива с числом даёт **булев массив** той же формы; им можно индексировать: `X[X < 0]` —
все отрицательные элементы, `X[X < 0] = 0` — обнулить их. Важная разница с Python: **срез массива —
это не копия, а представление** (view) тех же данных. `Y = X[:10]; Y[0] = 0` изменит и `X`.
Копию делают явно: `X.copy()`.

Напишите `clip_negative(X)`: верните **новый** массив, где отрицательные элементы заменены нулями,
и число таких элементов. Исходный массив не меняйте. Без циклов.

In [ ]:
def clip_negative(X):
    """Возвращает (новый массив без отрицательных элементов, число заменённых элементов)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
A = np.array([[1.0, -2.0], [-3.0, 4.0]])
A_copy = A.copy()
Y, count = clip_negative(A)
assert np.array_equal(Y, [[1, 0], [0, 4]]), f"Ожидалось [[1, 0], [0, 4]], получено {Y.tolist()}"
assert count == 2, f"Отрицательных элементов 2, а функция вернула {count}"
assert np.array_equal(A, A_copy), "Функция изменила исходный массив: срез или маска без .copy() — это тот же массив"
assert_no_loops(clip_negative)
print("OK")

## Задание 2.3. Индексация массивами: one-hot

Индексировать можно массивом индексов: `X[[2, 0]]` — строки 2 и 0; `X[rows, cols]` — элементы
`X[rows[0], cols[0]], X[rows[1], cols[1]], ...`. Это позволяет, например, в каждой строке выбрать
свой столбец.

Напишите `one_hot(y, K)`: по вектору меток `y` из $\{0, \dots, K-1\}$ длины $n$ постройте матрицу
$n \times K$ из нулей и единиц, в которой в строке $i$ единица стоит в столбце `y[i]`. Без циклов.

In [ ]:
def one_hot(y, K):
    """one_hot(np.array([2, 0]), 3) -> [[0, 0, 1], [1, 0, 0]]"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
E = one_hot(np.array([2, 0, 1, 2]), 3)
assert E.shape == (4, 3), f"Форма должна быть (4, 3), получено {E.shape}"
assert np.array_equal(E, [[0, 0, 1], [1, 0, 0], [0, 1, 0], [0, 0, 1]]), f"Неверная матрица: {E.tolist()}"
assert np.array_equal(one_hot(np.array([0]), 1), [[1]]), "Для y=[0], K=1 ожидается [[1]]"
assert_no_loops(one_hot)
print("OK")

## Задание 2.4. Попарные расстояния без циклов

Это главное задание части: из него вырастет метод ближайших соседей в практике 02.

Даны матрицы `A` формы `(m, d)` и `B` формы `(n, d)`. Нужно получить матрицу `D` формы `(m, n)`
квадратов евклидовых расстояний: `D[i, j]` $= \left\lVert \mathbf{a}_i - \mathbf{b}_j \right\rVert^2$. Без циклов.

Подсказка. Есть два пути:

1. broadcasting: `A[:, None, :] - B[None, :, :]` имеет форму `(m, n, d)` — остаётся возвести в квадрат
   и просуммировать по последней оси. Просто, но требует памяти $m \cdot n \cdot d$;
2. раскрыть квадрат: $\left\lVert \mathbf{a} - \mathbf{b} \right\rVert^2 = \left\lVert \mathbf{a} \right\rVert^2 + \left\lVert \mathbf{b} \right\rVert^2 - 2\,\mathbf{a}^{\top}\mathbf{b}$.
   Все скалярные произведения сразу — это `A @ B.T`, квадраты норм — `(A ** 2).sum(axis=1)`.
   Памяти нужно только $m \cdot n$. Из-за ошибок округления могут получиться маленькие отрицательные
   числа — обрежьте их нулём.

In [ ]:
def pairwise_sq_dists(A, B):
    """Матрица (m, n) квадратов евклидовых расстояний между строками A и строками B."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
A = rng.normal(size=(7, 4))
B = rng.normal(size=(5, 4))
D = pairwise_sq_dists(A, B)
reference = np.array([[np.sum((a - b) ** 2) for b in B] for a in A])
assert D.shape == (7, 5), f"Форма должна быть (7, 5), получено {D.shape}"
assert np.allclose(D, reference), "Расстояния не совпадают с посчитанными в лоб"
assert np.allclose(np.diag(pairwise_sq_dists(A, A)), 0), "Расстояние от точки до самой себя должно быть 0"
assert (pairwise_sq_dists(A, A) >= 0).all(), "Квадраты расстояний не могут быть отрицательными"
assert_no_loops(pairwise_sq_dists)
print("OK")

## Задание 2.5. Сортировка по строкам: k ближайших

`np.argsort(x)` возвращает индексы, упорядочивающие `x` по возрастанию. С параметром `axis=1` —
сортирует каждую строку матрицы независимо.

Напишите `k_nearest(D, k)`: по матрице расстояний `D` формы `(m, n)` верните массив формы `(m, k)`,
в строке `i` которого — индексы `k` самых маленьких элементов строки `D[i]`, по возрастанию
расстояния. Без циклов.

In [ ]:
def k_nearest(D, k):
    """k_nearest([[3, 1, 2]], 2) -> [[1, 2]]"""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
D = np.array([[3.0, 1.0, 2.0, 0.5], [0.0, 9.0, 8.0, 7.0]])
idx = k_nearest(D, 2)
assert idx.shape == (2, 2), f"Форма должна быть (2, 2), получено {np.shape(idx)}"
assert np.array_equal(idx, [[3, 1], [0, 3]]), f"Ожидалось [[3, 1], [0, 3]], получено {np.asarray(idx).tolist()}"
assert np.array_equal(k_nearest(D, 4)[1], [0, 3, 2, 1]), "Индексы должны идти по возрастанию расстояния"
assert_no_loops(k_nearest)
print("OK")

# Часть 3. pandas

`DataFrame` — таблица, у которой есть подписи строк (`index`) и столбцов (`columns`), а столбцы
могут иметь разные типы. Столбец — это `Series`. Основное:

| Операция | Пример |
|:---|:---|
| просмотр | `df.head()`, `df.info()`, `df.describe()`, `df.shape`, `df.dtypes` |
| столбцы | `df["age"]`, `df[["age", "sex"]]` |
| фильтрация | `df[df["age"] > 30]`, `df[(df.sex == "male") & (df.age < 18)]` |
| строки по меткам / позициям | `df.loc[10, "age"]`, `df.iloc[0:5]` |
| пропуски | `df.isna().sum()`, `df.fillna(...)`, `df.dropna()` |
| группировка | `df.groupby("pclass")["fare"].mean()`, `.agg(["mean", "median"])` |
| новый столбец | `df["family"] = df["sibsp"] + df["parch"] + 1` |

Логические условия объединяются операторами `&`, `|`, `~` (не `and`, `or`, `not`) и каждое берётся
в скобки. Как и в NumPy, циклы по строкам (`iterrows`) почти никогда не нужны.

Данные — пассажиры «Титаника» (1309 человек): класс билета `pclass`, пол `sex`, возраст `age`,
число братьев, сестёр и супругов на борту `sibsp`, родителей и детей `parch`, стоимость билета `fare`,
порт посадки `embarked` и целевая переменная `survived` (1 — выжил).

In [ ]:
# @title Загрузка данных: Titanic (OpenML, id 40945) { display-mode: "form" }
from sklearn.datasets import fetch_openml

titanic = fetch_openml(data_id=40945, as_frame=True, parser="auto").frame
# boat и body почти однозначно выдают ответ (номер шлюпки, номер найденного тела) — убираем.
titanic = titanic.drop(columns=["boat", "body", "home.dest", "ticket", "cabin"])
titanic["survived"] = titanic["survived"].astype(int)
titanic["sex"] = titanic["sex"].astype(str)
titanic["embarked"] = titanic["embarked"].astype(object)
print(titanic.shape)
titanic.head()

## Задание 3.1. Пропуски

Напишите `missing_report(df)`: верните `Series` с числом пропусков в каждом столбце, в котором они
есть, отсортированную по убыванию. Столбцы без пропусков в результат не включайте.

In [ ]:
def missing_report(df):
    """Series: столбец -> число пропусков (только > 0), по убыванию."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
report = missing_report(titanic)
assert isinstance(report, pd.Series), "Нужно вернуть pd.Series"
assert list(report.index) == ["age", "embarked", "fare"], f"Ожидались столбцы age, embarked, fare по убыванию, получено {list(report.index)}"
assert report["age"] == 263, f"Пропусков в age 263, получено {report['age']}"
assert missing_report(pd.DataFrame({"a": [1, 2]})).empty, "Если пропусков нет, результат должен быть пустым"
print("OK")

## Задание 3.2. Фильтрация и группировка

Напишите `survival_by_group(df)`: верните долю выживших (`survived` — это 0 и 1, так что доля равна
среднему) для каждой пары (пол, класс билета), только среди **взрослых** пассажиров (`age >= 18`;
пассажиры без возраста не учитываются). Результат — `Series` с двухуровневым индексом `(sex, pclass)`.

In [ ]:
def survival_by_group(df):
    """Series с индексом (sex, pclass): доля выживших среди взрослых."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
rates = survival_by_group(titanic)
assert isinstance(rates, pd.Series), "Нужно вернуть pd.Series"
assert rates.index.names == ["sex", "pclass"], f"Индекс должен быть (sex, pclass), получено {rates.index.names}"
assert len(rates) == 6, f"Групп (пол × класс) должно быть 6, получено {len(rates)}"
assert abs(rates[("female", 1)] - 0.968) < 1e-3, "Неверная доля выживших для женщин 1 класса: проверьте фильтр по возрасту"
assert abs(rates[("male", 3)] - 0.1557) < 1e-3, "Неверная доля выживших для мужчин 3 класса"
print("OK")

## Задание 3.3. Заполнение пропусков по группам

Заполнить пропущенный возраст одним общим средним грубо: дети из третьего класса и пожилые
пассажиры первого получат одно и то же значение. Лучше заполнять медианой **внутри группы**.
Для этого есть `groupby(...)[col].transform("median")`: результат имеет ту же длину, что и исходная
таблица, и в каждой строке — медиану её группы.

Напишите `fill_age(df)`: верните **копию** таблицы, где пропуски в `age` заполнены медианой возраста
по группе (пол, класс билета). Исходную таблицу не меняйте.

In [ ]:
def fill_age(df):
    """Копия df, в которой пропуски age заполнены медианой по (sex, pclass)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
before = titanic["age"].isna().sum()
filled = fill_age(titanic)
assert titanic["age"].isna().sum() == before, "Функция изменила исходную таблицу — работайте с копией (df.copy())"
assert filled["age"].isna().sum() == 0, "В age остались пропуски"
known = titanic["age"].notna()
assert (filled.loc[known, "age"] == titanic.loc[known, "age"]).all(), "Известные значения возраста не должны меняться"
i = titanic.index[titanic["age"].isna() & (titanic["sex"] == "male") & (titanic["pclass"] == 3)][0]
expected = titanic.loc[(titanic["sex"] == "male") & (titanic["pclass"] == 3), "age"].median()
assert filled.loc[i, "age"] == expected, "Пропуск у мужчины 3 класса должен заполниться медианой возраста мужчин 3 класса"
print("OK")

## Задание 3.4. Признаки для модели

Модели из scikit-learn принимают числовую матрицу `X` и вектор `y`. Категориальные признаки нужно
закодировать one-hot: `pd.get_dummies(df, columns=[...])` заменяет каждый указанный столбец набором
бинарных. Результат переводится в NumPy методом `.to_numpy()`.

Напишите `make_features(df)`, которая:

1. заполняет пропуски в `age` функцией `fill_age`, в `fare` — медианой `fare`, в `embarked` — самым
   частым значением (`df["embarked"].mode()[0]`);
2. добавляет признак `family = sibsp + parch + 1`;
3. оставляет признаки `pclass, sex, age, fare, family, embarked` и кодирует one-hot `sex` и `embarked`
   (`pclass` оставьте числом);
4. возвращает `X` (`np.ndarray` типа `float`), `y` (`np.ndarray` из `survived`) и список имён
   столбцов `X`.

In [ ]:
def make_features(df):
    """Возвращает (X: float ndarray, y: ndarray, feature_names: list[str])."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
X, y, names = make_features(titanic)
assert isinstance(X, np.ndarray) and X.dtype == float, "X должен быть массивом NumPy типа float"
assert X.shape == (1309, 9), f"Ожидалась форма (1309, 9): pclass, age, fare, family + 2 столбца пола + 3 порта; получено {X.shape}"
assert not np.isnan(X).any(), "В X остались пропуски"
assert len(names) == X.shape[1], "Число имён признаков должно совпадать с числом столбцов X"
assert "family" in names and "sex_male" in names, f"Среди признаков должны быть family и sex_male, получено {names}"
assert y.shape == (1309,) and set(np.unique(y)) == {0, 1}, "y должен быть вектором из 0 и 1 длины 1309"
j = names.index("family")
assert X[0, j] == titanic.loc[0, "sibsp"] + titanic.loc[0, "parch"] + 1, "Признак family посчитан неверно"
print("OK")

## Задание 3.5. Вопросы

Ответьте коротко, своими словами.

1. Почему в заданиях 2.2 и 3.3 пришлось явно делать копию, а в задании 2.1 — нет?
2. Во сколько раз векторизованное вычисление было быстрее цикла в начале части 2? Откуда такая разница?
3. Почему пол и порт посадки кодируются one-hot, а класс билета можно оставить числом?

*Ваш ответ:*